In [ ]:
import pandas as pd
# from LabData.DataLoaders.GutMBLoader import GutMBLoader
# from LabData.DataLoaders.SubjectLoader import SubjectLoader
# from LabData.DataLoaders.DietLoggingLoader import DietLoggingLoader
# from LabData.DataAnalyses.TenK_Trajectories.utils import get_diet_logging_around_stage
import seaborn as sns
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import pickle
import lightgbm as lgb
import math
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
from scipy.stats import spearmanr
import re
from LabData.DataLoaders.BodyMeasuresLoader import BodyMeasuresLoader
from LabData.DataLoaders.LifeStyleLoader import LifeStyleLoader
from LabData.DataLoaders.DemographicsLoader import DemographicsLoader
from LabData.DataLoaders.Medications10KLoader import Medications10KLoader



In [ ]:
home_path = '/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/'
SPECIES = 'segal_species' # 'segal_species' or 'mpa_species'
color1 = "#66C2A5"
single_style = "nature_single.mplstyle"
double_style = "nature_double.mplstyle"
third_style = "nature_third.mplstyle"
plt.rcParams["figure.dpi"] = 150
plt.style.use(single_style)
study_ids=[10, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010]
stage = 'baseline'

In [ ]:
def read_results(df):
    output = []
    for col in df.columns:
        output.append(df[col])
    return tuple(output)

In [ ]:
species = '' if SPECIES == 'segal_species' else '_mpa'

diet_mb = pd.read_pickle(home_path + f"data/{SPECIES}/diet_mb.pkl")
with open(home_path + f'data/{SPECIES}/my_lists.pkl', 'rb') as file:
    loaded_lists = pickle.load(file)
base_features, all_features, targets = loaded_lists
with open(home_path + f'data/{SPECIES}/scaler.pkl', 'rb') as scaler_file:
        scaler = pickle.load(scaler_file)
diet_mb

### BMI

In [ ]:
bml = BodyMeasuresLoader()
# bmld = bml.get_data(study_ids=study_ids, cols=['weight', 'height', 'bmr'])
bmld = bml.get_data(study_ids=study_ids)
bmldf = bmld.df
bmldf.info()

In [ ]:
if stage == 'baseline':
    # Keep only baseline
    bmldf = bmldf[~bmldf.index.get_level_values(0).duplicated()]
    bmldf = bmldf.reset_index(level=[1], drop=True)
elif stage == '02_00_visit':
    # Keep only the second entry (2nd visit)
    bmldf = bmldf.groupby(level=0).nth(1) 
elif stage == '04_00_visit':
    # Keep only the second entry (3rd visit)
    bmldf = bmldf.groupby(level=0).nth(1) 
bmldf

In [ ]:
diet_mb = diet_mb.join(bmldf['bmi'], how='inner')

### Comorbidities

In [ ]:
baseline_conditions = pd.read_csv("/net/mraid20/export/genie/LabData/Data/10K/for_review/baseline_conditions_all.csv")
follow_up_conditions = pd.read_csv("/net/mraid20/export/genie/LabData/Data/10K/for_review/follow_up_conditions_all.csv")
baseline_conditions

In [ ]:
baseline_conditions["Group"].value_counts()

In [ ]:
baseline_conditions["Consolidated name"].value_counts().head(30)

In [ ]:
baseline_conditions["research_stage"].value_counts().sort_index()

In [ ]:
follow_up_conditions["research_stage"].value_counts().sort_index()

In [ ]:
def get_condition_matrix_for_stages(baseline_conditions, follow_up_conditions):
    research_stages = ['baseline', '02_00_visit', '04_00_visit']
    selected_conditions = [
        "Obesity",
        "Prediabetes",
        "Hyperlipidemia",
        "Hypertension",
        "Fatty Liver Disease (NAFLD)",
        "Irritable Bowel Syndrome (IBS)",
        "Peptic Ulcer Disease",
        "Gallstone disease",
        "Asthma",
        "Atopic dermatitis",
        "Psoriasis",
        "Allergy",
        "Depression",
        "Anxiety",
        "Hypothyroidism"
    ]
    cond_cols = ["RegistrationCode", "research_stage", "Consolidated name"]
    full = pd.concat([
        baseline_conditions.loc[:, cond_cols],
        follow_up_conditions.loc[:, cond_cols]
    ], ignore_index=True)

    output = {}
    for stage in research_stages:
        all_rcs_in_stage = pd.concat([
            baseline_conditions[baseline_conditions['research_stage'] == stage]['RegistrationCode'],
            follow_up_conditions[follow_up_conditions['research_stage'] == stage]['RegistrationCode'],
        ]).drop_duplicates().sort_values().tolist()

        matrix = pd.DataFrame(0, index=all_rcs_in_stage, columns=selected_conditions)

        stage_df = full[(full["research_stage"] == stage) & (full["Consolidated name"].isin(selected_conditions))]
        for row in stage_df.itertuples(index=False, name=None):
            rc = row[0]
            cond = row[2]
            if rc in matrix.index and cond in matrix.columns:
                matrix.at[rc, cond] = 1

        matrix = matrix.astype(int)
        output[stage] = matrix

    return output

condition_matrices = get_condition_matrix_for_stages(baseline_conditions, follow_up_conditions)
condition_matrices_baseline = condition_matrices['baseline']
condition_matrices_02_00_visit = condition_matrices['02_00_visit']
condition_matrices_04_00_visit = condition_matrices['04_00_visit']


In [ ]:
# Helper to drop columns with no variation

def drop_no_variation(df: pd.DataFrame, name: str) -> pd.DataFrame:
    nunique = df.nunique()
    drop_cols = nunique[nunique <= 1].index.tolist()
    if drop_cols:
        print(f"Dropping {len(drop_cols)} no-variation columns from {name}: {drop_cols[:10]}")
        df = df.drop(columns=drop_cols)
    else:
        print(f"No no-variation columns in {name}")
    return df


In [ ]:
# Drop comorbidity columns: remove from ALL timepoints only if zero variance at baseline
zero_var_baseline_cols = condition_matrices_baseline.columns[condition_matrices_baseline.nunique() <= 1].tolist()
if zero_var_baseline_cols:
    print(f"Comorbidity zero-variance at baseline: {zero_var_baseline_cols[:10]}")
    print(f"Dropping {len(zero_var_baseline_cols)} comorbidity columns (zero variance at baseline)")
    condition_matrices_baseline = condition_matrices_baseline.drop(columns=zero_var_baseline_cols, errors='ignore')
    condition_matrices_02_00_visit = condition_matrices_02_00_visit.drop(columns=zero_var_baseline_cols, errors='ignore')
    condition_matrices_04_00_visit = condition_matrices_04_00_visit.drop(columns=zero_var_baseline_cols, errors='ignore')
    print(f"Dropped: {zero_var_baseline_cols[:10]}")
else:
    print("No comorbidity columns dropped for zero variance at baseline")


In [ ]:
condition_matrices_baseline

In [ ]:
condition_matrices_02_00_visit

In [ ]:
condition_matrices_04_00_visit

In [ ]:
diet_mb = diet_mb.join(condition_matrices_baseline, how='left')
diet_mb = diet_mb.fillna(0)
diet_mb.shape


### Lifestyle factors

In [ ]:
dl = DemographicsLoader()
dld = dl.get_data(study_ids=study_ids, df="english")
dldf = dld.df
list(dldf.columns)

In [ ]:
dldf.shape

In [ ]:
# dldf[['employment', 'living_place_today']].describe()
# dldf['employment'].value_counts().sort_index()
dldf['living_place_today'].value_counts().sort_index()

In [ ]:
dldf['living_place_today'].isna().sum()

In [ ]:
dldf

In [ ]:
import pandas as pd

# Ensure dldf index includes both RegistrationCode and Date as a MultiIndex (if not, set it)
if not isinstance(dldf.index, pd.MultiIndex) or dldf.index.names != ['RegistrationCode', 'Date']:
    if 'RegistrationCode' in dldf.columns and 'Date' in dldf.columns:
        dldf = dldf.set_index(['RegistrationCode', 'Date'])
    elif 'Date' in dldf.columns:
        dldf = dldf.set_index(['Date'])
    else:
        # Assume index is already at least RegistrationCode, set Date as second if available
        if 'Date' in dldf.columns:
            dldf = dldf.set_index('Date', append=True)
        # If not available, leave as is (shouldn't happen for correctly loaded dldf)

dldf = dldf.copy()

# Robustly ensure the 'Date' index is datetime64, even with duplicate values, to prevent ValueError
if 'Date' in dldf.index.names and not pd.api.types.is_datetime64_any_dtype(dldf.index.get_level_values('Date')):
    # Convert Date level in MultiIndex to datetime using repeatable, non-unique-safe method
    # Rebuild MultiIndex with Date coerced to datetime on a value-by-value basis
    idx = dldf.index
    # Get current levels
    regcode_level = idx.get_level_values('RegistrationCode')
    date_level = pd.to_datetime(idx.get_level_values('Date'), errors='coerce')
    # Build new MultiIndex with same tuples, but Date forced to datetime
    dldf.index = pd.MultiIndex.from_arrays([regcode_level, date_level], names=['RegistrationCode', 'Date'])

baseline_rows = []
baseline_dates = {}

for regcode, group in dldf.groupby(level=0):
    group_sorted = group.sort_index(level=1)
    first_row = group_sorted.iloc[[0]]
    baseline_rows.append(first_row)
    baseline_dates[regcode] = group_sorted.index.get_level_values(1)[0]

demog_baseline = pd.concat(baseline_rows)
demog_baseline_dates = pd.Series(baseline_dates, name='Baseline_Date')

# Prepare lists for 2y and 4y visits
visit_02_rows = []
visit_04_rows = []

# Define window in days
six_months_days = 182
two_years_days = 2 * 365
four_years_days = 4 * 365

for regcode, group in dldf.groupby(level=0):
    if regcode not in baseline_dates:
        continue
    baseline_date = baseline_dates[regcode]
    # Use .loc trick to always drop the right indices, even if there are duplicate dates
    group_other = group.copy()
    # Remove all rows with baseline_date (may be more than one)
    if (group_other.index.get_level_values(1) == baseline_date).any():
        mask = group_other.index.get_level_values(1) != baseline_date
        group_other = group_other[mask]
    if group_other.empty:
        continue
    date_deltas = group_other.index.get_level_values(1) - baseline_date
    delta_days = date_deltas.days

    mask_02 = ((delta_days >= (two_years_days - six_months_days)) &
               (delta_days <= (two_years_days + six_months_days)))
    visit_02 = group_other[mask_02]
    if not visit_02.empty:
        visit_02_rows.append(visit_02)

    mask_04 = ((delta_days >= (four_years_days - six_months_days)) &
               (delta_days <= (four_years_days + six_months_days)))
    visit_04 = group_other[mask_04]
    if not visit_04.empty:
        visit_04_rows.append(visit_04)

if visit_02_rows:
    demog_02_visit = pd.concat(visit_02_rows)
else:
    demog_02_visit = pd.DataFrame(columns=dldf.columns)

if visit_04_rows:
    demog_04_visit = pd.concat(visit_04_rows)
else:
    demog_04_visit = pd.DataFrame(columns=dldf.columns)


In [ ]:
demog_baseline = demog_baseline[["employment", "living_place_today", "total_income"]].reset_index(drop=True, level=1)
demog_02_visit = demog_02_visit[["employment", "living_place_today", "total_income"]].reset_index(drop=True, level=1)
demog_04_visit = demog_04_visit[["employment", "living_place_today", "total_income"]].reset_index(drop=True, level=1)

In [ ]:
demog_02_visit

In [ ]:
print("Baseline duplicates:", demog_baseline.index.duplicated().any())
print("Visit 02 duplicates:", demog_02_visit.index.duplicated().any())
# Remove duplicates from the original DataFrames (keeping the first occurrence)
demog_baseline = demog_baseline.loc[~demog_baseline.index.duplicated(keep='first')]
demog_02_visit = demog_02_visit.loc[~demog_02_visit.index.duplicated(keep='first')]
demog_04_visit = demog_04_visit.loc[~demog_04_visit.index.duplicated(keep='first')]

In [ ]:
common_ids = demog_baseline.index.intersection(demog_02_visit.index)
demog_02_visit_change = (demog_02_visit.loc[common_ids] != demog_baseline.loc[common_ids]).astype(int)

common_ids = demog_04_visit.index.intersection(demog_02_visit.index)
demog_04_visit_change = (demog_04_visit.loc[common_ids] != demog_02_visit.loc[common_ids]).astype(int)

# Drop columns with no variance across ALL demog change dataframes (baseline is all zeros)
combined_demog_changes = pd.concat([demog_02_visit_change, demog_04_visit_change], axis=0)
cols_drop_demog = combined_demog_changes.columns[combined_demog_changes.nunique() <= 1].tolist()
if cols_drop_demog:
    print(f"Dropping {len(cols_drop_demog)} no-variation demog change columns (across baseline/02/04): {cols_drop_demog[:10]}")
    demog_02_visit_change = demog_02_visit_change.drop(columns=cols_drop_demog)
    demog_04_visit_change = demog_04_visit_change.drop(columns=cols_drop_demog)
else:
    print("No demog change columns dropped for no variation across time points")

In [ ]:
print(demog_02_visit_change.shape)
print(demog_04_visit_change.shape)
demog_02_visit_change.tail()
demog_04_visit_change.tail()

In [ ]:
demog_02_visit_change

In [ ]:
print(demog_02_visit_change.sum(axis=0))
print(demog_02_visit_change.shape)
print(demog_04_visit_change.sum(axis=0))
print(demog_04_visit_change.shape)

##### Lifestyle Loader

In [ ]:
lll = LifeStyleLoader()
llld = lll.get_data(study_ids=study_ids)
llldf = llld.df
# list(llldf.columns)

In [ ]:
list(llldf.columns)

In [ ]:
llldf

In [ ]:
import pandas as pd

# Assume llldf is already loaded and has MultiIndex (RegistrationCode, Date)
# Ensure Date index is datetime
llldf = llldf.copy()
if not pd.api.types.is_datetime64_any_dtype(llldf.index.get_level_values("Date")):
    llldf.index = llldf.index.set_levels(
        pd.to_datetime(llldf.index.levels[1]), level=1
    )

# Find baseline (first) date for each RegistrationCode
baseline_rows = []
baseline_dates = {}

for regcode, group in llldf.groupby(level=0):
    group_sorted = group.sort_index(level=1)
    first_row = group_sorted.iloc[[0]]
    baseline_rows.append(first_row)
    baseline_dates[regcode] = group_sorted.index.get_level_values(1)[0]

lifestyle_baseline = pd.concat(baseline_rows)
baseline_date_series = pd.Series(baseline_dates, name="Baseline_Date")

# Prepare empty lists to collect 02 and 04 visit rows
visit_02_rows = []
visit_04_rows = []

# Define window (in days)
six_months_days = 182  # ~6 months
two_years_days = 2 * 365
four_years_days = 4 * 365

for regcode, group in llldf.groupby(level=0):
    if regcode not in baseline_dates:
        continue  # shouldn't happen
    baseline_date = baseline_dates[regcode]
    # Exclude baseline row itself (by date)
    group_other = group.drop(baseline_date, level=1, errors="ignore")
    if group_other.empty:
        continue
    # Compute delta days for each row from baseline
    date_deltas = group_other.index.get_level_values(1) - baseline_date
    delta_days = date_deltas.days

    # Find rows within 2y +/-6m (i.e., [2y-6m, 2y+6m])
    mask_02 = ((delta_days >= (two_years_days - six_months_days)) &
               (delta_days <= (two_years_days + six_months_days)))
    visit_02 = group_other[mask_02]
    if not visit_02.empty:
        visit_02_rows.append(visit_02)

    # Find rows within 4y +/-6m
    mask_04 = ((delta_days >= (four_years_days - six_months_days)) &
               (delta_days <= (four_years_days + six_months_days)))
    visit_04 = group_other[mask_04]
    if not visit_04.empty:
        visit_04_rows.append(visit_04)

if visit_02_rows:
    lifestyle_02_visit = pd.concat(visit_02_rows)
else:
    lifestyle_02_visit = pd.DataFrame(columns=llldf.columns)

if visit_04_rows:
    lifestyle_04_visit = pd.concat(visit_04_rows)
else:
    lifestyle_04_visit = pd.DataFrame(columns=llldf.columns)

# Reset index for baseline for consistency (optional), or keep as MultiIndex if preferred
lifestyle_baseline = lifestyle_baseline.copy()



In [ ]:
lifestyle_baseline

In [ ]:
print(lifestyle_02_visit.shape)
print(lifestyle_04_visit.shape)


In [ ]:
def process_lifestyle_data(lifestyle_df: pd.DataFrame) -> pd.DataFrame:
    """
    Processes the lifestyle DataFrame:
    - Removes the 'Date' index, keeping only 'RegistrationCode'
    - Selects several direct columns
    - Aggregates smoking columns into binary columns
    - Aggregates household relationship columns into grouped binary columns
    - Normalizes all -1 values to 0
    - Fills NaN values in pet_present and accommodation_type with 0

    Args:
        lifestyle_df: The input DataFrame with MultiIndex ('RegistrationCode', 'Date')

    Returns:
        A new DataFrame with processed columns, indexed by RegistrationCode
    """
    # Remove 'Date' index, keep only RegistrationCode
    if isinstance(lifestyle_df.index, pd.MultiIndex):
        lifestyle_df = lifestyle_df.reset_index(level='Date', drop=True)

    result_cols = [
        'pet_present',
        'accommodation_type',
    ]
    df_result = lifestyle_df[result_cols].copy()

    # Fill pet_present and accommodation_type NaNs with 0
    for col in ['pet_present', 'accommodation_type']:
        if col in df_result.columns:
            df_result[col] = df_result[col].fillna(0)

    # --- Smoking columns ---
    current_smoker_col = 'smoke_tobacco_now'
    if current_smoker_col in lifestyle_df.columns:
        df_result['is_current_smoker'] = lifestyle_df[current_smoker_col].fillna(0).astype(int)
    else:
        df_result['is_current_smoker'] = 0

    household_smoker_col = 'smoke_houshold'
    if household_smoker_col in lifestyle_df.columns:
        df_result['is_household_smoker'] = lifestyle_df[household_smoker_col].fillna(0).astype(int)
    else:
        df_result['is_household_smoker'] = 0

    # --- Relationship columns aggregation ---
    living_with_relative_cols = [
        'people_living_together_retalated__Another connection',
        'people_living_together_retalated__Brother, sister or both',
        'people_living_together_retalated__Grandchildren',
        'people_living_together_retalated__Grandparents',
        'people_living_together_retalated__Son, daughter or both',
        'people_living_together_retalated__parents'
    ]
    living_with_relative_cols_present = [c for c in living_with_relative_cols if c in lifestyle_df.columns]
    if living_with_relative_cols_present:
        df_result['living_with_relative'] = lifestyle_df[living_with_relative_cols_present].fillna(0).max(axis=1).astype(int)
    else:
        df_result['living_with_relative'] = 0

    # Spouse/partner (treated separately from other unrelated)
    living_with_spouse_cols = [
        'people_living_together_retalated__Husband, wife or spouse',
        'people_living_together_retalated__Partner'
    ]
    living_with_spouse_cols_present = [c for c in living_with_spouse_cols if c in lifestyle_df.columns]
    if living_with_spouse_cols_present:
        df_result['living_with_spouse'] = lifestyle_df[living_with_spouse_cols_present].fillna(0).max(axis=1).astype(int)
    else:
        df_result['living_with_spouse'] = 0

    # Other not-related (excluding spouse/partner)
    living_with_not_related_cols = [
        'people_living_together_retalated__Other',
        'people_living_together_retalated__Other unrelated'
    ]
    living_with_not_related_cols_present = [c for c in living_with_not_related_cols if c in lifestyle_df.columns]
    if living_with_not_related_cols_present:
        df_result['living_with_not_related'] = lifestyle_df[living_with_not_related_cols_present].fillna(0).max(axis=1).astype(int)
    else:
        df_result['living_with_not_related'] = 0

    # Normalize -1 to 0 across all columns
    df_result = df_result.replace(-1, 0)

    return df_result

lifestyle_baseline = process_lifestyle_data(lifestyle_baseline)
print(lifestyle_baseline.head())

In [ ]:
lifestyle_02_visit = process_lifestyle_data(lifestyle_02_visit)
print(lifestyle_02_visit.head())
lifestyle_04_visit = process_lifestyle_data(lifestyle_04_visit)
print(lifestyle_04_visit.head())

In [ ]:
lifestyle_04_visit.isna().sum()
lifestyle_02_visit.isna().sum()

In [ ]:
for col in lifestyle_baseline.columns:
    print(col)
    print(lifestyle_baseline[col].value_counts())
    print('-'*100)

In [ ]:
print(lifestyle_baseline.info())
print(lifestyle_02_visit.info())
print(lifestyle_04_visit.info())


In [ ]:
print("Baseline duplicates:", lifestyle_baseline.index.duplicated().any())
print("Visit 02 duplicates:", lifestyle_02_visit.index.duplicated().any())

In [ ]:
# Remove duplicates from the original DataFrames (keeping the first occurrence)
lifestyle_baseline = lifestyle_baseline.loc[~lifestyle_baseline.index.duplicated(keep='first')]
lifestyle_02_visit = lifestyle_02_visit.loc[~lifestyle_02_visit.index.duplicated(keep='first')]
lifestyle_04_visit = lifestyle_04_visit.loc[~lifestyle_04_visit.index.duplicated(keep='first')]

In [ ]:
common_ids = lifestyle_baseline.index.intersection(lifestyle_02_visit.index)
lifestyle_02_visit_change = (lifestyle_02_visit.loc[common_ids] != lifestyle_baseline.loc[common_ids]).astype(int)
lifestyle_02_visit_change = drop_no_variation(lifestyle_02_visit_change, "lifestyle_02_visit_change")

common_ids = lifestyle_04_visit.index.intersection(lifestyle_02_visit.index)
lifestyle_04_visit_change = (lifestyle_04_visit.loc[common_ids] != lifestyle_02_visit.loc[common_ids]).astype(int)
lifestyle_04_visit_change = drop_no_variation(lifestyle_04_visit_change, "lifestyle_04_visit_change")

In [ ]:
print(lifestyle_02_visit_change.shape)
print(lifestyle_04_visit_change.shape)

In [ ]:
lifestyle_04_visit_change.tail()

In [ ]:
print(lifestyle_02_visit_change.sum(axis=0))
print(lifestyle_02_visit_change.shape)
print(lifestyle_04_visit_change.sum(axis=0))
print(lifestyle_04_visit_change.shape)

#### Adam and Gil's datasets

In [ ]:
# lifestyle_baseline = pd.read_csv("/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/10K_Trajectories/body_systems/lifestyle_baseline.csv")
# lifestyle_02_visit = pd.read_csv("/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/10K_Trajectories/body_systems/lifestyle_02_00_visit.csv")
# lifestyle_04_visit = pd.read_csv("/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/10K_Trajectories/body_systems/lifestyle_04_00_visit.csv")
# lifestyle_baseline

In [ ]:
# def process_lifestyle_data(lifestyle_df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Selects 'pet_present_yes' and combines smoking-related columns 
#     into two new binary columns. Combination means 1 if any component is 1, else 0.

#     Args:
#         lifestyle_df: The input DataFrame containing the lifestyle columns.

#     Returns:
#         A new DataFrame with the three processed columns.
#     """
    
#     # 1. Select the direct column
#     lifestyle_df.set_index('RegistrationCode', inplace=True)
#     df_result = lifestyle_df[['pet_present_yes']].copy()
    
#     # 2. Combine columns for person currently smokes
#     current_smoker_cols = [
#         'smoke_tobacco_now_Only sometimes', 
#         'smoke_tobacco_now_Yes  most or all days'
#     ]
    
#     # Use max(axis=1) for the OR operation: 1 if any column is 1
#     df_result['is_current_smoker'] = lifestyle_df[current_smoker_cols].max(axis=1)
    
#     # 3. Combine columns for someone in the household smokes
#     household_smoker_cols = [
#         'smoke_houshold_Yes  more than one household member smokes', 
#         'smoke_houshold_Yes  one household member smokes'
#     ]
    
#     # Use max(axis=1) for the OR operation: 1 if any column is 1
#     df_result['is_household_smoker'] = lifestyle_df[household_smoker_cols].max(axis=1)
    
#     return df_result


# lifestyle_baseline = process_lifestyle_data(lifestyle_baseline)
# print(lifestyle_baseline.head())

In [ ]:
# lifestyle_02_visit = process_lifestyle_data(lifestyle_02_visit)
# print(lifestyle_02_visit.head())
# lifestyle_04_visit = process_lifestyle_data(lifestyle_04_visit)
# print(lifestyle_04_visit.head())

In [ ]:
# lifestyle_04_visit.isna().sum()
# lifestyle_02_visit.isna().sum()

In [ ]:
# common_ids = lifestyle_baseline.index.intersection(lifestyle_02_visit.index)
# lifestyle_02_visit_change = lifestyle_02_visit.loc[common_ids] - lifestyle_baseline.loc[common_ids]
# common_ids = lifestyle_04_visit.index.intersection(lifestyle_02_visit.index)
# lifestyle_04_visit_change = lifestyle_04_visit.loc[common_ids] - lifestyle_02_visit.loc[common_ids]

In [ ]:
# lifestyle_02_visit_change.shape

### Medications

In [ ]:
# # Load medications data
# ml = Medications10KLoader()
# mld = ml.get_data(study_ids=study_ids, pivot_by=3)
# mldf = mld.df
# list(mldf.columns)

In [ ]:
medications_baseline = pd.read_csv("/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/10K_Trajectories/body_systems/medications_baseline.csv")
medications_02_visit = pd.read_csv("/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/10K_Trajectories/body_systems/medications_02_00_visit.csv")
medications_04_visit = pd.read_csv("/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/10K_Trajectories/body_systems/medications_04_00_visit.csv")
medications_baseline

In [ ]:
# Ensure lifestyle change columns have variation across all time points (baseline is zeros)
combined_lifestyle_changes = pd.concat([lifestyle_02_visit_change, lifestyle_04_visit_change], axis=0)
cols_drop_lifestyle = combined_lifestyle_changes.columns[combined_lifestyle_changes.nunique() <= 1].tolist()
if cols_drop_lifestyle:
    print(f"Dropping {len(cols_drop_lifestyle)} no-variation lifestyle change columns (across baseline/02/04): {cols_drop_lifestyle[:10]}")
    lifestyle_02_visit_change = lifestyle_02_visit_change.drop(columns=cols_drop_lifestyle)
    lifestyle_04_visit_change = lifestyle_04_visit_change.drop(columns=cols_drop_lifestyle)
else:
    print("No lifestyle change columns dropped for no variation across time points")


In [ ]:
# Find columns (medications) present in all three DataFrames
common_columns = set(medications_baseline.columns) & set(medications_02_visit.columns) & set(medications_04_visit.columns)
common_columns = list(common_columns)

# Subset the DataFrames to these columns
mb = medications_baseline[common_columns]
m2 = medications_02_visit[common_columns]
m4 = medications_04_visit[common_columns]

# Compute prevalence for each medication in each dataset
prevalence_baseline = mb.mean()
prevalence_02_visit = m2.mean()
prevalence_04_visit = m4.mean()

valid_range_baseline = (prevalence_baseline > 0.05) & (prevalence_baseline < 0.95)
valid_range_02_visit = (prevalence_02_visit > 0.05) & (prevalence_02_visit < 0.95)
valid_range_04_visit = (prevalence_04_visit > 0.05) & (prevalence_04_visit < 0.95)

# Find columns with True in all three datasets
cols_in_valid_range_all = valid_range_baseline & valid_range_02_visit & valid_range_04_visit
cols_in_valid_range_all = cols_in_valid_range_all[cols_in_valid_range_all].index.tolist()

print("Columns with prevalence in (0.05, 0.95) for all three datasets:", cols_in_valid_range_all)

In [ ]:
print((prevalence_baseline > 0.05) & (prevalence_baseline < 0.95))
print((prevalence_02_visit > 0.05) & (prevalence_02_visit < 0.95))
print((prevalence_04_visit > 0.05) & (prevalence_04_visit < 0.95))


In [ ]:
# For each medications dataset, subset to cols_in_valid_range_all and set RegistrationCode as index
mb_sub = medications_baseline[['RegistrationCode'] + cols_in_valid_range_all].set_index('RegistrationCode')
m2_sub = medications_02_visit[['RegistrationCode'] + cols_in_valid_range_all].set_index('RegistrationCode')
m4_sub = medications_04_visit[['RegistrationCode'] + cols_in_valid_range_all].set_index('RegistrationCode')


In [ ]:
# For each subject, create a DataFrame showing medication changes between visits
# Change from baseline to 02 visit
common_ids_b2 = mb_sub.index.intersection(m2_sub.index)
medications_02_visit_change = (m2_sub.loc[common_ids_b2] != mb_sub.loc[common_ids_b2]).astype(int)

# Change from 02 to 04 visit
common_ids_24 = m2_sub.index.intersection(m4_sub.index)
medications_04_visit_change = (m4_sub.loc[common_ids_24] != m2_sub.loc[common_ids_24]).astype(int)

# Drop columns with no variance across ALL medication change dataframes (baseline is all zeros)
combined_med_changes = pd.concat([medications_02_visit_change, medications_04_visit_change], axis=0)
cols_drop_med = combined_med_changes.columns[combined_med_changes.nunique() <= 1].tolist()
if cols_drop_med:
    print(f"Dropping {len(cols_drop_med)} no-variation medications change columns (across baseline/02/04): {cols_drop_med[:10]}")
    medications_02_visit_change = medications_02_visit_change.drop(columns=cols_drop_med)
    medications_04_visit_change = medications_04_visit_change.drop(columns=cols_drop_med)
else:
    print("No medications change columns dropped for no variation across time points")


In [ ]:
medications_02_visit_change

In [ ]:
medications_04_visit_change

In [ ]:
# # Create a version of diet_mb with Fatty Liver Disease (NAFLD) column included (filled with zeros)
# # This is for a colleague who needs the column even though it has zero variance at baseline
# diet_mb_with_fatty_liver = pd.read_pickle(home_path + f"data/{SPECIES}/diet_mb.pkl")

# # Add Fatty Liver Disease (NAFLD) column with all zeros
# fatty_liver_col_name = "Fatty Liver Disease (NAFLD)"
# diet_mb_with_fatty_liver[fatty_liver_col_name] = 0

# # Save with a new name
# output_path = home_path + f"data/{SPECIES}/diet_mb_with_fatty_liver.pkl"
# diet_mb_with_fatty_liver.to_pickle(output_path)
# print(f"Saved diet_mb with Fatty Liver column to: {output_path}")
# print(f"Shape: {diet_mb_with_fatty_liver.shape}")
# print(f"Fatty Liver column present: {fatty_liver_col_name in diet_mb_with_fatty_liver.columns}")


### Saving

In [ ]:
bmi = "bmi"
comorbidities = condition_matrices_baseline.columns
lifestyle = lifestyle_baseline.columns

In [ ]:
# Add covariates to 02/04 visit datasets and save
# Read visit datasets similar to train_models.py
pathways = ''  # keep empty to mirror current files
CLR_suf = ''   # no CLR suffix in this notebook

# Load visit diet_mb tables
visit_02_path = home_path + f"data/{SPECIES}/diet_mb{pathways}_02_visit{CLR_suf}.pkl"
visit_04_path = home_path + f"data/{SPECIES}/diet_mb{pathways}_04_visit{CLR_suf}.pkl"
diet_mb_02_visit = pd.read_pickle(visit_02_path)
diet_mb_04_visit = pd.read_pickle(visit_04_path)

# Helper to select BMI per stage (same logic as baseline block above)
def select_bmi_for_stage(df, stage_label):
    if stage_label == 'baseline':
        out = df[~df.index.get_level_values(0).duplicated()]
        out = out.reset_index(level=[1], drop=True)
    elif stage_label == '02_00_visit':
        out = df.groupby(level=0).nth(1)
    elif stage_label == '04_00_visit':
        out = df.groupby(level=0).nth(1)
    else:
        out = df
    return out['bmi']

# Reuse the original body measures dataframe
bmldf_full = bml.get_data(study_ids=study_ids).df
bmi_02 = select_bmi_for_stage(bmldf_full.copy(), '02_00_visit')
bmi_04 = select_bmi_for_stage(bmldf_full.copy(), '04_00_visit')

# Join BMI
diet_mb_02_visit = diet_mb_02_visit.join(bmi_02, how='left')
diet_mb_04_visit = diet_mb_04_visit.join(bmi_04, how='left')

# Join comorbidities for the matching stage and fill missing with 0
if 'condition_matrices_02_00_visit' in globals():
    diet_mb_02_visit = diet_mb_02_visit.join(condition_matrices_02_00_visit, how='left')
    diet_mb_02_visit = diet_mb_02_visit.fillna(0)

if 'condition_matrices_04_00_visit' in globals():
    diet_mb_04_visit = diet_mb_04_visit.join(condition_matrices_04_00_visit, how='left')
    diet_mb_04_visit = diet_mb_04_visit.fillna(0)

# Save the visit datasets with covariates
visit_02_out = home_path + f"data/{SPECIES}/diet_mb{pathways}_02_visit{CLR_suf}.pkl"
visit_04_out = home_path + f"data/{SPECIES}/diet_mb{pathways}_04_visit{CLR_suf}.pkl"
diet_mb_02_visit.to_pickle(visit_02_out)
diet_mb_04_visit.to_pickle(visit_04_out)

print("Saved:", visit_02_out)
print("Saved:", visit_04_out)


In [ ]:
print(diet_mb_02_visit.shape)
print(diet_mb_04_visit.shape)

In [ ]:
diet_mb.to_pickle(home_path + f"data/{SPECIES}/diet_mb.pkl")
with open(home_path + f'data/covariates.pkl', 'wb') as file:
    pickle.dump([bmi, comorbidities, lifestyle], file)

# Save my_lists.pkl with base_features+all covariates and all_features+all covariates
# Assumes base_features and all_features exist, and "covariates" means bmi+comorbidities+lifestyle columns as vector
all_covariate_cols = []
all_covariate_cols.append(bmi)
all_covariate_cols.extend(list(comorbidities))
# all_covariate_cols.extend(list(lifestyle))

base_features_with_covs = list(base_features) + all_covariate_cols
all_features_with_covs = list(all_features) + all_covariate_cols
print(base_features_with_covs)
with open(home_path + f'data/{SPECIES}/my_lists.pkl', 'wb') as file:
    pickle.dump([base_features_with_covs, all_features_with_covs, targets], file)

In [ ]:
# Save all changes dataframes
# Create baseline changes (all zeros, same structure as other changes)
# Demographics baseline changes
if 'demog_baseline' in globals() and 'demog_02_visit_change' in globals():
    demog_baseline_change = pd.DataFrame(
        0, 
        index=demog_baseline.index, 
        columns=demog_02_visit_change.columns
    )
    demog_baseline_change.to_pickle(home_path + f"data/demog_baseline_change.pkl")
    demog_02_visit_change.to_pickle(home_path + f"data/demog_02_visit_change.pkl")
    demog_04_visit_change.to_pickle(home_path + f"data/demog_04_visit_change.pkl")
    demog_change_cols = list(demog_02_visit_change.columns)
    print("Saved demographics changes")

# Lifestyle baseline changes
if 'lifestyle_baseline' in globals() and 'lifestyle_02_visit_change' in globals():
    lifestyle_baseline_change = pd.DataFrame(
        0, 
        index=lifestyle_baseline.index, 
        columns=lifestyle_02_visit_change.columns
    )
    lifestyle_baseline_change.to_pickle(home_path + f"data/lifestyle_baseline_change.pkl")
    lifestyle_02_visit_change.to_pickle(home_path + f"data/lifestyle_02_visit_change.pkl")
    lifestyle_04_visit_change.to_pickle(home_path + f"data/lifestyle_04_visit_change.pkl")
    lifestyle_change_cols = list(lifestyle_02_visit_change.columns)
    print("Saved lifestyle changes")

# Medications baseline changes
if 'mb_sub' in globals() and 'medications_02_visit_change' in globals():
    medications_baseline_change = pd.DataFrame(
        0, 
        index=mb_sub.index, 
        columns=medications_02_visit_change.columns
    )
    medications_baseline_change.to_pickle(home_path + f"data/medications_baseline_change.pkl")
    medications_02_visit_change.to_pickle(home_path + f"data/medications_02_visit_change.pkl")
    medications_04_visit_change.to_pickle(home_path + f"data/medications_04_visit_change.pkl")
    medications_change_cols = list(medications_02_visit_change.columns)
    print("Saved medications changes")

# Save column lists for change dataframes
if 'demog_change_cols' in globals() and 'lifestyle_change_cols' in globals() and 'medications_change_cols' in globals():
    with open(home_path + 'data/change_covariates.pkl', 'wb') as file:
        pickle.dump([demog_change_cols, lifestyle_change_cols, medications_change_cols], file)
    print("Saved change covariates column lists")
